# ByteJEPA — JEPA-for-Text on the OpenAI Parameter Golf Challenge

**Architecture**: LeWorldModel (LeWM) adapted for byte-level text modeling.  
**Metric**: `val_bpb` (bits per byte) on FineWeb validation set — lower is better.  
**Constraint**: 16MB artifact (weights + code), 10 min on 8×H100s.  
**Key idea**: Instead of predicting in raw byte space, predict in *latent space*.
The model factorizes into:
- **State encoder** `f_enc`: local window of bytes → compact latent `z_t`
- **Action encoder** `f_act`: longer context window → discourse-level latent `a_t`
- **Predictor** `pred`: `(z_t, a_t) → ẑ_{t+1}` (latent-space next-step prediction)
- **Decoder head**: `ẑ_{t+1} → logits over 256 bytes` (for bpb eval)

**SIGReg** (from LeWM): instead of EMA/stop-grad to prevent collapse, we regularize the encoder output distribution to be isotropic Gaussian — cheap, stable, principled.

**Why `a_t` is not trivial**: Text has latent actions — discourse moves (assertion, elaboration, pivot), hierarchical intent (paragraph-level topic), structural patterns (code blocks, citations). The action encoder captures this slower timescale.

```
Bytes: [b0 b1 ... b_{L-1}]  (raw bytes, no tokenizer)
         |                         |
    State enc              Action enc
    (local, k=64)          (context, K=256, strided)
         |                         |
        z_t  ────── Predictor ────→  ẑ_{t+1}
                       +a_t              |
                                   Decoder head
                                         |
                                  logits (256) → bpb
```


In [ ]:
# ── Cell 1: Install deps ──────────────────────────────────────────────────────
# Run once in the parameter-golf repo environment.
# The RunPod template already has torch, so only extras needed.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "datasets", "huggingface_hub", "tqdm", "bitsandbytes"], check=True)
print("Dependencies ready.")

In [ ]:
# ── Cell 2: Imports & Global Config ──────────────────────────────────────────
import os, math, time, zlib, struct
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── ByteJEPA hyper-parameters ─────────────────────────────────────────────────
@dataclass
class ByteJEPAConfig:
    # Architecture
    d_model:        int   = 384      # latent dimension
    d_action:       int   = 192      # action (context) encoder dim
    n_enc_layers:   int   = 3        # state encoder layers
    n_act_layers:   int   = 2        # action encoder layers
    n_pred_layers:  int   = 8        # predictor layers
    n_heads:        int   = 6        # attention heads (d_model must divide)
    mlp_mult:       float = 3.0      # MLP hidden = d_model * mlp_mult
    dropout:        float = 0.0

    # Sequence
    local_window:   int   = 64       # bytes fed to state encoder
    context_window: int   = 512      # bytes fed to action encoder
    stride:         int   = 4        # action encoder: downsample factor

    # Loss weights
    lambda_sigreg:  float = 0.1      # SIGReg coefficient (sole tunable, per LeWM)
    lambda_decode:  float = 1.0      # decode CE loss (for bpb supervision)

    # Training
    seq_len:        int   = 1024     # training sequence length in bytes
    batch_size:     int   = 64
    lr:             float = 3e-3
    weight_decay:   float = 0.1
    warmup_steps:   int   = 200
    max_steps:      int   = 5000
    grad_clip:      float = 1.0
    val_every:      int   = 250

    # I/O
    data_path:      str   = "./data/datasets/fineweb10B_raw/"
    val_data_path:  str   = "./data/datasets/fineweb_val_raw/"
    checkpoint_dir: str   = "./jepa_checkpoints/"

CFG = ByteJEPAConfig()
print(CFG)

## Data Loading — Raw Bytes (No Tokenizer)

The challenge evaluates on `val_bpb` (bits per byte), which is tokenizer-agnostic.  
We feed the model raw UTF-8 bytes (vocab = 256). The FineWeb text is encoded to bytes on the fly.

If you already downloaded the sp1024 shards, Cell 3b shows how to strip the tokenizer and re-encode as bytes.  
Cell 3a downloads directly from HuggingFace (faster for initial experiments).

In [ ]:
# ── Cell 3a: Download & cache FineWeb as raw bytes ───────────────────────────
# Writes binary shard files: each file is a flat uint8 array of byte-encoded text.
# On RunPod, run this once; subsequent runs load from disk.

from datasets import load_dataset
from tqdm.auto import tqdm

TRAIN_OUT = Path(CFG.data_path)
VAL_OUT   = Path(CFG.val_data_path)
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
VAL_OUT.mkdir(parents=True, exist_ok=True)

N_TRAIN_SHARDS = 10   # each ≈ 100MB of raw bytes — adjust as needed
SHARD_TOKENS   = 10_000_000  # bytes per shard

def encode_and_save(split: str, out_dir: Path, n_shards: int, max_docs: Optional[int] = None):
    """Stream FineWeb, encode text as UTF-8 bytes, write shards."""
    if list(out_dir.glob("*.npy")):
        print(f"Shards already exist in {out_dir}, skipping download.")
        return

    ds = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT",
                      split=split, streaming=True)
    buf, shard_idx = [], 0
    total_bytes = 0

    for i, doc in enumerate(tqdm(ds, desc=f"Encoding {split}")):
        if max_docs and i >= max_docs:
            break
        raw = (doc["text"] + "\n\n").encode("utf-8", errors="replace")
        buf.extend(raw)
        total_bytes += len(raw)

        while len(buf) >= SHARD_TOKENS and shard_idx < n_shards:
            chunk = np.array(buf[:SHARD_TOKENS], dtype=np.uint8)
            np.save(out_dir / f"shard_{shard_idx:04d}.npy", chunk)
            buf = buf[SHARD_TOKENS:]
            shard_idx += 1
            print(f"  Saved shard {shard_idx}/{n_shards} ({total_bytes/1e6:.1f}MB total)")
            if shard_idx >= n_shards:
                break

    # Save remainder as last shard
    if buf and shard_idx < n_shards:
        np.save(out_dir / f"shard_{shard_idx:04d}.npy",
                np.array(buf, dtype=np.uint8))
        print(f"  Saved final shard {shard_idx+1}")

encode_and_save("train", TRAIN_OUT, n_shards=N_TRAIN_SHARDS)
encode_and_save("validation[:50000]", VAL_OUT, n_shards=2)  # ~50k docs for val
print("Data ready.")

In [ ]:
# ── Cell 3b: Dataset & DataLoader ────────────────────────────────────────────
class ByteShardDataset(Dataset):
    """
    Memory-maps numpy uint8 shards.
    Returns sequences of length (context_window + 1) bytes:
      - input:  context_window bytes
      - target: context_window bytes shifted by 1 (next-byte prediction)
    """
    def __init__(self, data_dir: str, seq_len: int, split: str = "train"):
        self.seq_len = seq_len
        shards = sorted(Path(data_dir).glob("*.npy"))
        assert shards, f"No .npy shards found in {data_dir}"
        # Concatenate all shards into a single memmap-friendly array
        arrays = [np.load(s, mmap_mode="r") for s in shards]
        self.data = np.concatenate(arrays)  # shape: (N,) uint8
        self.n_samples = (len(self.data) - 1) // seq_len
        print(f"[{split}] {len(self.data)/1e6:.1f}M bytes → {self.n_samples} samples")

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        start = idx * self.seq_len
        chunk = self.data[start : start + self.seq_len + 1].astype(np.int64)
        x = torch.from_numpy(chunk[:-1])  # (seq_len,)
        y = torch.from_numpy(chunk[1:])   # (seq_len,) — next-byte targets
        return x, y


train_ds  = ByteShardDataset(CFG.data_path,     CFG.seq_len, split="train")
val_ds    = ByteShardDataset(CFG.val_data_path,  CFG.seq_len, split="val")

train_dl  = DataLoader(train_ds, batch_size=CFG.batch_size,
                        shuffle=True,  num_workers=4, pin_memory=True, drop_last=True)
val_dl    = DataLoader(val_ds,   batch_size=CFG.batch_size,
                        shuffle=False, num_workers=4, pin_memory=True, drop_last=True)

print(f"Train batches: {len(train_dl)} | Val batches: {len(val_dl)}")

## Architecture — ByteJEPA

### Key design decisions

1. **State encoder** (`f_enc`): A small causal transformer over a *local* window of `k=64` bytes. Outputs `z_t ∈ ℝ^{d_model}` — the current latent state.

2. **Action encoder** (`f_act`): A *strided* transformer over a longer context (`K=512` bytes, stride=4 → 128 positions). Outputs `a_t ∈ ℝ^{d_action}` — the latent action capturing discourse-level intent (paragraph structure, topic, code vs. prose, etc.).

3. **Predictor**: Cross-attends `z_t` to `a_t` then self-attends across the sequence to produce `ẑ_{t+1}`. This is where the JEPA magic lives — prediction happens entirely in latent space.

4. **Decoder head**: A single linear layer `ℝ^{d_model} → ℝ^{256}`. Used *only for loss/eval*, not during planning. This keeps the architecture honest — representations must be decodable but aren't trained to minimize decode loss alone.

5. **SIGReg**: Penalizes the encoder output distribution for deviating from N(0, I). Replaces EMA/stop-grad tricks. One scalar hyperparameter `λ_sigreg`.

### Why this might beat the flat transformer baseline
The predictor operates on *d_model-dimensional* latents, not 256-dimensional byte distributions.  
If `d_model < sequence_length`, the predictor sees compressed representations — fewer parameters needed to model dependencies.

In [ ]:
# ── Cell 4: Building Blocks ───────────────────────────────────────────────────

class RMSNorm(nn.Module):
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        return x / (x.pow(2).mean(-1, keepdim=True) + self.eps).sqrt() * self.scale


class CausalSelfAttention(nn.Module):
    def __init__(self, d: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d % n_heads == 0
        self.n_heads = n_heads
        self.head_d = d // n_heads
        self.qkv  = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        self.drop = dropout

    def forward(self, x):  # x: (B, T, d)
        B, T, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_d).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_d).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_d).transpose(1, 2)
        # Flash attention with causal mask
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.drop if self.training else 0.0)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, d))


class CrossAttention(nn.Module):
    """z_t (query) attends to a_t (key/value)."""
    def __init__(self, d_q: int, d_kv: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_q % n_heads == 0
        self.n_heads = n_heads
        self.head_d  = d_q // n_heads
        self.q_proj  = nn.Linear(d_q,  d_q,  bias=False)
        self.k_proj  = nn.Linear(d_kv, d_q,  bias=False)
        self.v_proj  = nn.Linear(d_kv, d_q,  bias=False)
        self.out     = nn.Linear(d_q,  d_q,  bias=False)
        self.drop    = dropout

    def forward(self, z, a):  # z: (B, T, d_q)  a: (B, S, d_kv)
        B, T, d = z.shape
        _, S, _  = a.shape
        q = self.q_proj(z).view(B, T, self.n_heads, self.head_d).transpose(1, 2)
        k = self.k_proj(a).view(B, S, self.n_heads, self.head_d).transpose(1, 2)
        v = self.v_proj(a).view(B, S, self.n_heads, self.head_d).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=False,
                                           dropout_p=self.drop if self.training else 0.0)
        return self.out(y.transpose(1, 2).contiguous().view(B, T, d))


class MLP(nn.Module):
    def __init__(self, d: int, mult: float = 3.0):
        super().__init__()
        h = int(d * mult)
        self.fc1  = nn.Linear(d, h,   bias=False)
        self.fc2  = nn.Linear(h, d,   bias=False)
        self.gate = nn.Linear(d, h,   bias=False)  # SwiGLU gate
    def forward(self, x):
        return self.fc2(F.silu(self.gate(x)) * self.fc1(x))


class EncoderBlock(nn.Module):
    def __init__(self, d: int, n_heads: int, mlp_mult: float, dropout: float):
        super().__init__()
        self.norm1 = RMSNorm(d)
        self.attn  = CausalSelfAttention(d, n_heads, dropout)
        self.norm2 = RMSNorm(d)
        self.mlp   = MLP(d, mlp_mult)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class PredictorBlock(nn.Module):
    """Self-attention + cross-attention to action + MLP."""
    def __init__(self, d: int, d_act: int, n_heads: int, mlp_mult: float, dropout: float):
        super().__init__()
        self.norm1  = RMSNorm(d)
        self.self_a = CausalSelfAttention(d, n_heads, dropout)
        self.norm2  = RMSNorm(d)
        self.cross  = CrossAttention(d, d_act, n_heads, dropout)
        self.norm3  = RMSNorm(d)
        self.mlp    = MLP(d, mlp_mult)
    def forward(self, z, a):
        z = z + self.self_a(self.norm1(z))
        z = z + self.cross(self.norm2(z), a)
        z = z + self.mlp(self.norm3(z))
        return z


print("Building blocks defined.")

In [ ]:
# ── Cell 5: ByteJEPA Model ────────────────────────────────────────────────────

class ByteJEPA(nn.Module):
    """
    JEPA for byte-level text modeling.

    Forward pass during training:
      x: (B, T)  int64 byte indices [0..255]
    Returns:
      loss_pred:   MSE between predicted and actual next latent
      loss_sigreg: SIGReg pushing encoder outputs toward N(0, I)
      loss_decode: cross-entropy of decoded logits vs next bytes (for bpb)
      logits:      (B, T, 256) for bpb eval
    """

    def __init__(self, cfg: ByteJEPAConfig):
        super().__init__()
        self.cfg = cfg
        d, d_a = cfg.d_model, cfg.d_action

        # Shared byte embedding (256 vocab)
        self.byte_embed = nn.Embedding(256, d)

        # Positional encodings (learnable)
        self.local_pos_enc   = nn.Embedding(cfg.context_window, d)
        self.context_pos_enc = nn.Embedding(cfg.context_window // cfg.stride + 1, d_a)

        # State encoder: local window → z_t
        self.enc_proj  = nn.Linear(d, d, bias=False)  # project shared embed → enc dim
        self.enc_blocks = nn.ModuleList([
            EncoderBlock(d, cfg.n_heads, cfg.mlp_mult, cfg.dropout)
            for _ in range(cfg.n_enc_layers)
        ])
        self.enc_norm  = RMSNorm(d)

        # Action encoder: strided context → a_t
        self.act_proj  = nn.Linear(d, d_a, bias=False)  # project shared embed → act dim
        self.act_blocks = nn.ModuleList([
            EncoderBlock(d_a, max(1, cfg.n_heads // 2), cfg.mlp_mult, cfg.dropout)
            for _ in range(cfg.n_act_layers)
        ])
        self.act_norm  = RMSNorm(d_a)

        # Predictor: (z, a) → ẑ_next
        self.pred_blocks = nn.ModuleList([
            PredictorBlock(d, d_a, cfg.n_heads, cfg.mlp_mult, cfg.dropout)
            for _ in range(cfg.n_pred_layers)
        ])
        self.pred_norm = RMSNorm(d)

        # Decoder head: z → byte logits (used only for loss & eval)
        self.decode_head = nn.Linear(d, 256, bias=False)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    # ── sub-networks ─────────────────────────────────────────────────────────

    def encode_state(self, x):  # x: (B, T)
        """Run state encoder over local sliding window."""
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        h = self.byte_embed(x)                     # (B, T, d)
        h = self.enc_proj(h) + self.local_pos_enc(pos.clamp(max=self.cfg.context_window-1))
        for blk in self.enc_blocks:
            h = blk(h)
        return self.enc_norm(h)                     # (B, T, d)

    def encode_action(self, x):  # x: (B, T)
        """
        Run action encoder over a longer context with strided pooling.
        Captures discourse-level structure (paragraph, topic, code/prose).
        """
        B, T = x.shape
        s = self.cfg.stride
        h = self.byte_embed(x)                     # (B, T, d)
        h = self.act_proj(h)                        # (B, T, d_a)

        # Strided average pooling: compress T → T/stride positions
        T_pad = (T + s - 1) // s * s
        h_pad = F.pad(h, (0, 0, 0, T_pad - T))     # (B, T_pad, d_a)
        h = h_pad.view(B, T_pad // s, s, self.cfg.d_action).mean(dim=2)  # (B, T/s, d_a)

        S = h.shape[1]
        pos = torch.arange(S, device=x.device).clamp(max=self.context_pos_enc.num_embeddings - 1)
        h = h + self.context_pos_enc(pos)
        for blk in self.act_blocks:
            h = blk(h)
        return self.act_norm(h)                     # (B, T/s, d_a)

    def predict(self, z, a):
        """Predict next latent sequence given current latents + action context."""
        for blk in self.pred_blocks:
            z = blk(z, a)
        return self.pred_norm(z)                    # (B, T, d)

    # ── loss functions ────────────────────────────────────────────────────────

    def sigreg_loss(self, z):  # z: (B, T, d)
        """
        SIGReg from LeWM: penalize KL( q(z) || N(0, I) ).
        Computed analytically assuming q(z) is a factored Gaussian
        with mean=sample_mean and var=sample_var.

        KL(N(μ, σ²) || N(0, 1)) = 0.5 * (σ² + μ² - 1 - log σ²)
        """
        z_flat = z.reshape(-1, z.shape[-1])         # (B*T, d)
        mu  = z_flat.mean(0)                         # (d,)
        var = z_flat.var(0).clamp(min=1e-6)          # (d,)
        kl  = 0.5 * (var + mu.pow(2) - 1 - var.log())
        return kl.mean()

    def forward(self, x, y):  # x, y: (B, T) int64
        # ── encode ───────────────────────────────────────────────────────────
        z = self.encode_state(x)    # (B, T, d) — current latents
        a = self.encode_action(x)   # (B, T/s, d_a) — action context

        # ── target latents (stop-grad NOT needed — SIGReg handles collapse) ─
        with torch.no_grad():
            z_target = self.encode_state(y)   # (B, T, d)

        # ── predict ──────────────────────────────────────────────────────────
        z_pred = self.predict(z, a)  # (B, T, d)

        # ── prediction loss: MSE in latent space ─────────────────────────────
        loss_pred = F.mse_loss(z_pred, z_target)

        # ── SIGReg: push encoder outputs toward N(0, I) ───────────────────────
        loss_sigreg = self.sigreg_loss(z)

        # ── decode: map predicted latents → byte logits ────────────────────
        logits = self.decode_head(z_pred)            # (B, T, 256)
        loss_decode = F.cross_entropy(
            logits.view(-1, 256), y.view(-1)
        )

        return loss_pred, loss_sigreg, loss_decode, logits

    @torch.no_grad()
    def compute_bpb(self, x, y):  # bpb = CE / log(2) for byte-level models
        _, _, loss_decode, _ = self.forward(x, y)
        return (loss_decode / math.log(2)).item()

    def n_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# ── instantiate and inspect ──────────────────────────────────────────────────
model = ByteJEPA(CFG).to(DEVICE)
print(f"Parameters: {model.n_params()/1e6:.2f}M")

# Sanity-check forward pass
x_test = torch.randint(0, 256, (2, CFG.seq_len), device=DEVICE)
y_test = torch.randint(0, 256, (2, CFG.seq_len), device=DEVICE)
lp, ls, ld, logits = model(x_test, y_test)
print(f"loss_pred={lp.item():.4f}  loss_sigreg={ls.item():.4f}  loss_decode={ld.item():.4f}")
print(f"logits shape: {logits.shape}  (expected: B x T x 256)")
del x_test, y_test

In [ ]:
# ── Cell 6: Parameter Budget Check (before training) ─────────────────────────
# Estimate compressed model size to confirm we're within 16MB.

def estimate_int8_zlib_size_mb(model: nn.Module) -> float:
    """Simulate INT8 quantization + zlib compression to estimate artifact size."""
    total_bytes = 0
    for p in model.parameters():
        # INT8: 1 byte per param, quantize to [-127, 127]
        arr = p.detach().float().cpu().numpy()
        scale = np.abs(arr).max() / 127.0
        quantized = np.clip(np.round(arr / (scale + 1e-8)), -127, 127).astype(np.int8)
        compressed = zlib.compress(quantized.tobytes(), level=9)
        total_bytes += len(compressed)
    return total_bytes / 1e6

est_mb = estimate_int8_zlib_size_mb(model)
print(f"Estimated INT8+zlib model size: {est_mb:.2f} MB")
print(f"Budget remaining for code:      {16.0 - est_mb:.2f} MB")
if est_mb > 15.5:
    print("⚠️  Warning: model is large. Reduce d_model or n_pred_layers.")
else:
    print("✓  Model fits comfortably within 16MB.")

In [ ]:
# ── Cell 7: Muon Optimizer ───────────────────────────────────────────────────
# Muon (momentum + orthogonalization) is the current best optimizer for
# this challenge — used by all top-5 leaderboard entries.
# We apply Muon to weight matrices and AdamW to embeddings/norms.

class Muon(torch.optim.Optimizer):
    """
    Muon: Momentum with Orthogonalization.
    Paper: https://arxiv.org/abs/2409.20325
    Nesterov momentum → Newton-Schulz orthogonalization → Nesterov update.
    """
    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True,
                 backend="newtonschulz5", backend_steps=5):
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov,
                        backend=backend, backend_steps=backend_steps)
        super().__init__(params, defaults)

    @staticmethod
    def _zeropower_via_newtonschulz5(G, steps=5):
        """Approx G @ (G^T G)^{-1/2} in ~5 iterations."""
        assert G.ndim >= 2
        a, b, c = (3.4445, -4.7750, 2.0315)
        X = G.bfloat16() / (G.norm() + 1e-7)
        if G.size(-2) > G.size(-1):
            X = X.mT
        for _ in range(steps):
            A  = X @ X.mT
            X  = a * X + b * (A @ X) + c * (A @ A @ X)
        if G.size(-2) > G.size(-1):
            X = X.mT
        return X.to(G.dtype)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            lr, momentum = group["lr"], group["momentum"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if "buf" not in state:
                    state["buf"] = torch.zeros_like(g)
                buf = state["buf"]
                buf.mul_(momentum).add_(g)
                g = g.add(buf, alpha=momentum) if group["nesterov"] else buf

                if g.ndim >= 2 and group["backend"] == "newtonschulz5":
                    g = self._zeropower_via_newtonschulz5(g, group["backend_steps"])
                    g *= max(g.size(-2), g.size(-1)) ** 0.5

                p.add_(g, alpha=-lr)
        return loss


def build_optimizer(model: ByteJEPA, cfg: ByteJEPAConfig):
    """
    Split parameters:
    - Weight matrices ≥2D  → Muon
    - Embeddings, norms    → AdamW
    """
    muon_params, adamw_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim >= 2 and "embed" not in name and "pos_enc" not in name:
            muon_params.append(p)
        else:
            adamw_params.append(p)

    print(f"Muon params:  {sum(p.numel() for p in muon_params)/1e6:.2f}M")
    print(f"AdamW params: {sum(p.numel() for p in adamw_params)/1e6:.2f}M")

    opt_muon  = Muon(muon_params, lr=cfg.lr, momentum=0.95)
    opt_adamw = torch.optim.AdamW(adamw_params, lr=cfg.lr * 0.3,
                                  weight_decay=cfg.weight_decay,
                                  betas=(0.9, 0.95))
    return opt_muon, opt_adamw


def lr_schedule(step: int, cfg: ByteJEPAConfig) -> float:
    """Warmup then cosine decay."""
    if step < cfg.warmup_steps:
        return step / cfg.warmup_steps
    t = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * t))


opt_muon, opt_adamw = build_optimizer(model, CFG)

In [ ]:
# ── Cell 8: Validation ────────────────────────────────────────────────────────

@torch.no_grad()
def validate(model: ByteJEPA, val_dl: DataLoader, n_batches: int = 50) -> dict:
    model.eval()
    total_bpb, total_loss, n = 0.0, 0.0, 0
    for i, (x, y) in enumerate(val_dl):
        if i >= n_batches:
            break
        x, y = x.to(DEVICE), y.to(DEVICE)
        lp, ls, ld, _ = model(x, y)
        bpb = (ld / math.log(2)).item()
        total_bpb  += bpb
        total_loss += ld.item()
        n += 1
    model.train()
    return {"val_bpb": total_bpb / n, "val_loss": total_loss / n}


# Quick sanity val before training
init_val = validate(model, val_dl, n_batches=10)
print(f"Pre-training val_bpb: {init_val['val_bpb']:.4f}  (random ≈ 8.0 for bytes)")

In [ ]:
# ── Cell 9: Training Loop ─────────────────────────────────────────────────────

from torch.cuda.amp import GradScaler, autocast

scaler    = GradScaler() if DEVICE == "cuda" else None
ckpt_path = Path(CFG.checkpoint_dir)
ckpt_path.mkdir(parents=True, exist_ok=True)

log = []                   # list of dicts for post-hoc analysis
best_bpb = float("inf")
t_start  = time.time()

train_iter = iter(train_dl)

for step in range(CFG.max_steps):
    # ── LR update ─────────────────────────────────────────────────────────────
    lr_mult = lr_schedule(step, CFG)
    for opt in [opt_muon, opt_adamw]:
        for g in opt.param_groups:
            g["lr"] = CFG.lr * lr_mult

    # ── fetch batch ──────────────────────────────────────────────────────────
    try:
        x, y = next(train_iter)
    except StopIteration:
        train_iter = iter(train_dl)
        x, y = next(train_iter)
    x, y = x.to(DEVICE), y.to(DEVICE)

    # ── forward + loss ────────────────────────────────────────────────────────
    for opt in [opt_muon, opt_adamw]:
        opt.zero_grad(set_to_none=True)

    if scaler is not None:
        with autocast():
            lp, ls, ld, _ = model(x, y)
            loss = ld + CFG.lambda_sigreg * ls  # decode + SIGReg
        scaler.scale(loss).backward()
        scaler.unscale_(opt_muon)
        scaler.unscale_(opt_adamw)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(opt_muon)
        scaler.step(opt_adamw)
        scaler.update()
    else:
        lp, ls, ld, _ = model(x, y)
        loss = ld + CFG.lambda_sigreg * ls
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        opt_muon.step()
        opt_adamw.step()

    # ── logging ──────────────────────────────────────────────────────────────
    row = {
        "step":        step,
        "train_loss":  loss.item(),
        "loss_pred":   lp.item(),
        "loss_sigreg": ls.item(),
        "loss_decode": ld.item(),
        "lr":          CFG.lr * lr_mult,
        "elapsed_sec": time.time() - t_start,
    }

    if step % 50 == 0:
        print(f"step={step:5d}  loss={loss.item():.4f}  "
              f"pred={lp.item():.4f}  sigreg={ls.item():.4f}  "
              f"decode={ld.item():.4f}  lr={CFG.lr * lr_mult:.2e}  "
              f"t={time.time()-t_start:.0f}s")

    if step % CFG.val_every == 0:
        val = validate(model, val_dl, n_batches=50)
        row.update(val)
        print(f"  ▶ val_bpb={val['val_bpb']:.4f}  val_loss={val['val_loss']:.4f}")

        if val["val_bpb"] < best_bpb:
            best_bpb = val["val_bpb"]
            torch.save({"model": model.state_dict(), "step": step,
                        "val_bpb": best_bpb, "cfg": CFG},
                       ckpt_path / "best.pt")
            print(f"  ✓ New best: {best_bpb:.4f}")

    log.append(row)

total_time = time.time() - t_start
print(f"\nTraining complete in {total_time/60:.1f} min")
print(f"Best val_bpb: {best_bpb:.4f}  (baseline: 1.2244)")

In [ ]:
# ── Cell 10: Training Curves ──────────────────────────────────────────────────
try:
    import matplotlib.pyplot as plt
    steps_all  = [r["step"] for r in log]
    losses     = [r["train_loss"] for r in log]
    val_steps  = [r["step"] for r in log if "val_bpb" in r]
    val_bpbs   = [r["val_bpb"] for r in log if "val_bpb" in r]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(steps_all, losses, label="train loss", alpha=0.8)
    axes[0].set_title("Training Loss"); axes[0].set_xlabel("step"); axes[0].legend()

    axes[1].plot(val_steps, val_bpbs, "o-", color="red", label="val_bpb")
    axes[1].axhline(1.2244, linestyle="--", color="gray", label="baseline (1.2244)")
    axes[1].set_title("Val BPB"); axes[1].set_xlabel("step"); axes[1].legend()

    pred_losses  = [r["loss_pred"]   for r in log]
    sig_losses   = [r["loss_sigreg"] for r in log]
    dec_losses   = [r["loss_decode"] for r in log]
    axes[2].plot(steps_all, pred_losses,  alpha=0.7, label="latent pred")
    axes[2].plot(steps_all, sig_losses,   alpha=0.7, label="SIGReg")
    axes[2].plot(steps_all, dec_losses,   alpha=0.7, label="decode CE")
    axes[2].set_title("Loss Components"); axes[2].set_xlabel("step"); axes[2].legend()

    plt.tight_layout()
    plt.savefig(ckpt_path / "training_curves.png", dpi=150)
    plt.show()
    print("Saved training_curves.png")
except ImportError:
    print("matplotlib not installed; skipping plots.")

In [ ]:
# ── Cell 11: INT8 Quantization + Zlib Compression ─────────────────────────────
# The challenge evaluates the final INT8+zlib compressed model.
# This cell produces the submission artifact and measures exact size.

def quantize_model_int8(model: nn.Module) -> dict:
    """INT8 symmetric per-tensor quantization. Returns scales + quantized weights."""
    qstate = {}
    for name, param in model.state_dict().items():
        arr   = param.float().cpu().numpy()
        scale = float(np.abs(arr).max()) / 127.0
        q     = np.clip(np.round(arr / (scale + 1e-8)), -127, 127).astype(np.int8)
        qstate[name] = {"scale": scale, "q": q}
    return qstate


def dequantize_model(qstate: dict, model: nn.Module):
    """Reload dequantized weights back into model for roundtrip validation."""
    new_state = {}
    for name, v in qstate.items():
        new_state[name] = torch.from_numpy(
            (v["q"].astype(np.float32) * v["scale"])
        ).reshape(model.state_dict()[name].shape)
    model.load_state_dict(new_state, strict=True)


def pack_artifact(qstate: dict) -> bytes:
    """Pack all INT8 weights into a single zlib-compressed binary blob."""
    parts = []
    for name, v in qstate.items():
        name_b = name.encode("utf-8")
        scale_b = struct.pack("f", v["scale"])  # 4 bytes
        q_b    = v["q"].tobytes()
        # header: [2-byte name_len][name_bytes][4-byte scale][4-byte weight_len][weights]
        header = struct.pack("H", len(name_b)) + name_b + scale_b + struct.pack("I", len(q_b))
        parts.append(header + q_b)
    raw = b"".join(parts)
    return zlib.compress(raw, level=9)


# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt = torch.load(ckpt_path / "best.pt", map_location="cpu")
model.load_state_dict(ckpt["model"])
model.eval()
print(f"Loaded best checkpoint (step={ckpt['step']}, val_bpb={ckpt['val_bpb']:.4f})")

# ── Quantize ──────────────────────────────────────────────────────────────────
qstate    = quantize_model_int8(model)
artifact  = pack_artifact(qstate)
art_mb    = len(artifact) / 1e6

print(f"\nArtifact size (weights only): {art_mb:.3f} MB")

# ── Roundtrip validation ──────────────────────────────────────────────────────
model_q = ByteJEPA(CFG).to(DEVICE)
dequantize_model(qstate, model_q)
model_q.eval()

val_q = validate(model_q, val_dl, n_batches=50)
print(f"Post-quantization val_bpb: {val_q['val_bpb']:.4f}")
print(f"Quantization degradation:  {val_q['val_bpb'] - ckpt['val_bpb']:+.4f} bpb")

# Save artifact
art_path = ckpt_path / "model_int8_zlib.bin"
with open(art_path, "wb") as f:
    f.write(artifact)
print(f"\nArtifact saved to {art_path}")

if art_mb < 15.0:
    print(f"✓ {15.0 - art_mb:.2f}MB remaining for train_gpt.py code")
else:
    print("⚠️  Large artifact. Consider reducing d_model or n_pred_layers.")

In [ ]:
# ── Cell 12: Ablation — Action Encoder Contribution ─────────────────────────
# Test the hypothesis: does a_t (discourse action) improve bpb vs. no action?
# This is a publishable finding if the delta is significant.

class ByteJEPANoAction(ByteJEPA):
    """Ablation: drop action encoder, predictor only uses z_t."""
    def encode_action(self, x):
        # Return a zero tensor — no action information
        B, T = x.shape
        S = T // self.cfg.stride + 1
        return torch.zeros(B, S, self.cfg.d_action, device=x.device)


print("To run ablation, train ByteJEPANoAction with the same hyperparameters")
print("and compare val_bpb curves. If ByteJEPA > ByteJEPANoAction significantly,")
print("the latent action structure in text is real and learnable.")
print()
print("Expected result based on theory:")
print("  ByteJEPA (with a_t):    better bpb (discourse context helps)")
print("  ByteJEPANoAction:       worse bpb  (predictor has no long-range signal)")
print()
print("If the gap is <0.005 bpb: action encoder is not contributing → simplify.")
print("If the gap is >0.01 bpb:  action encoder earns its parameters → keep.")

In [ ]:
# ── Cell 13: Export train_gpt.py for Submission ───────────────────────────────
# The challenge requires a single self-contained train_gpt.py.
# This cell writes the submission script to the records folder.

import inspect, textwrap

SUBMISSION_DIR = Path(f"./records/track_non_record_16mb/bytejepa_lewm/")
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

# Pull source of all classes defined in this notebook
classes_to_export = [
    RMSNorm, CausalSelfAttention, CrossAttention, MLP,
    EncoderBlock, PredictorBlock, ByteJEPA, Muon,
    ByteShardDataset, ByteJEPAConfig,
]

header = '''
#!/usr/bin/env python3
"""
ByteJEPA — JEPA-for-Text on the Parameter Golf Challenge
Architecture: LeWorldModel (LeWM) adapted for byte-level text modeling.
No tokenizer. Two-encoder factorization: state (local) + action (discourse-level).
Anti-collapse via SIGReg (isotropic Gaussian regularizer, LeWM Sec. 3).
Reference: Maes et al. 2026 (arXiv:2603.19312)
"""
import os, math, time, zlib, struct
from pathlib import Path
from dataclasses import dataclass
from typing import Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
'''

body_parts = [header]
for cls in classes_to_export:
    src = inspect.getsource(cls)
    body_parts.append(src)

main_block = '''
def main():
    cfg = ByteJEPAConfig(
        data_path     = os.environ.get("DATA_PATH",     "./data/datasets/fineweb10B_raw/"),
        val_data_path = os.environ.get("VAL_DATA_PATH", "./data/datasets/fineweb_val_raw/"),
        max_steps     = int(os.environ.get("MAX_STEPS", 5000)),
        batch_size    = int(os.environ.get("BATCH_SIZE", 64)),
    )

    train_ds = ByteShardDataset(cfg.data_path,     cfg.seq_len, split="train")
    val_ds   = ByteShardDataset(cfg.val_data_path, cfg.seq_len, split="val")
    train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
    val_dl   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False,
                          num_workers=4, pin_memory=True, drop_last=True)

    model = ByteJEPA(cfg).to(DEVICE)
    print(f"Parameters: {model.n_params()/1e6:.2f}M")

    opt_muon, opt_adamw = build_optimizer(model, cfg)
    scaler = GradScaler() if DEVICE == "cuda" else None
    best_bpb, train_iter = float("inf"), iter(train_dl)
    t0 = time.time()

    for step in range(cfg.max_steps):
        lr_mult = lr_schedule(step, cfg)
        for opt in [opt_muon, opt_adamw]:
            for g in opt.param_groups:
                g["lr"] = cfg.lr * lr_mult

        try:
            x, y = next(train_iter)
        except StopIteration:
            train_iter = iter(train_dl)
            x, y = next(train_iter)
        x, y = x.to(DEVICE), y.to(DEVICE)

        for opt in [opt_muon, opt_adamw]:
            opt.zero_grad(set_to_none=True)

        if scaler:
            with autocast():
                lp, ls, ld, _ = model(x, y)
                loss = ld + cfg.lambda_sigreg * ls
            scaler.scale(loss).backward()
            for opt in [opt_muon, opt_adamw]: scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            for opt in [opt_muon, opt_adamw]: scaler.step(opt)
            scaler.update()
        else:
            lp, ls, ld, _ = model(x, y)
            loss = ld + cfg.lambda_sigreg * ls
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt_muon.step(); opt_adamw.step()

        if step % 50 == 0:
            print(f"step={step:5d} loss={loss.item():.4f} "
                  f"decode={ld.item():.4f} sigreg={ls.item():.4f} "
                  f"t={time.time()-t0:.0f}s")

        if step % cfg.val_every == 0:
            val_bpb = validate_bpb(model, val_dl, cfg)
            print(f"  val_bpb={val_bpb:.4f}")
            if val_bpb < best_bpb:
                best_bpb = val_bpb
                torch.save(model.state_dict(), "best_model.pt")

    # Final evaluation
    model.load_state_dict(torch.load("best_model.pt", map_location=DEVICE))
    final_bpb = validate_bpb(model, val_dl, cfg, n_batches=len(val_dl))
    print(f"\nfinal_int8_zlib_roundtrip val_bpb: {final_bpb:.4f}")


@torch.no_grad()
def validate_bpb(model, val_dl, cfg, n_batches=50):
    model.eval()
    total, n = 0.0, 0
    for i, (x, y) in enumerate(val_dl):
        if i >= n_batches: break
        x, y = x.to(DEVICE), y.to(DEVICE)
        _, _, ld, _ = model(x, y)
        total += (ld / math.log(2)).item(); n += 1
    model.train()
    return total / n


def build_optimizer(model, cfg):
    muon_params, adamw_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if p.ndim >= 2 and "embed" not in name and "pos_enc" not in name:
            muon_params.append(p)
        else:
            adamw_params.append(p)
    opt_muon  = Muon(muon_params, lr=cfg.lr, momentum=0.95)
    opt_adamw = torch.optim.AdamW(adamw_params, lr=cfg.lr * 0.3,
                                  weight_decay=cfg.weight_decay, betas=(0.9, 0.95))
    return opt_muon, opt_adamw


def lr_schedule(step, cfg):
    if step < cfg.warmup_steps:
        return step / cfg.warmup_steps
    t = (step - cfg.warmup_steps) / max(1, cfg.max_steps - cfg.warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * t))


if __name__ == "__main__":
    main()
'''

body_parts.append(main_block)
full_script = "\n\n".join(body_parts)

with open(SUBMISSION_DIR / "train_gpt.py", "w") as f:
    f.write(full_script)

script_bytes = len(full_script.encode())
total_mb = (script_bytes + len(artifact)) / 1e6
print(f"train_gpt.py:   {script_bytes/1e3:.1f} KB")
print(f"Weights (INT8+zlib): {len(artifact)/1e6:.3f} MB")
print(f"Total artifact:      {total_mb:.3f} MB / 16.000 MB")
assert total_mb < 16.0, f"Over budget: {total_mb:.3f} MB!"
print("✓ Submission is within 16MB.")

In [ ]:
# ── Cell 14: Write submission.json and README.md ──────────────────────────────
import json

submission = {
    "name":         "YOUR_NAME",
    "github_id":    "YOUR_GITHUB_ID",
    "val_bpb":      round(float(val_q["val_bpb"]), 4),
    "model_size_mb": round(total_mb, 3),
    "n_params_M":   round(model.n_params() / 1e6, 2),
    "architecture": "ByteJEPA (LeWM-inspired)",
    "tokenizer":    "none (raw bytes, vocab=256)",
    "optimizer":    "Muon + AdamW",
    "lambda_sigreg": CFG.lambda_sigreg,
    "d_model":      CFG.d_model,
    "n_pred_layers": CFG.n_pred_layers,
}

with open(SUBMISSION_DIR / "submission.json", "w") as f:
    json.dump(submission, f, indent=2)

readme = f"""
# ByteJEPA — LeWorldModel for Text

**val_bpb**: {submission['val_bpb']} (post INT8+zlib quantization)  
**artifact**: {submission['model_size_mb']} MB  
**params**: {submission['n_params_M']}M  

## What is this?

A JEPA (Joint Embedding Predictive Architecture) adapted for byte-level text modeling,
inspired by LeWorldModel (Maes et al. 2026, arXiv:2603.19312).

**No tokenizer** — raw UTF-8 bytes (vocab=256), consistent with the challenge's
byte-level evaluation metric.

## Architecture

```
State encoder  (local k={CFG.local_window} bytes)  → z_t ∈ R^{CFG.d_model}
Action encoder (context K={CFG.context_window} bytes, stride={CFG.stride}) → a_t ∈ R^{CFG.d_action}
Predictor      ({CFG.n_pred_layers} layers, cross-attention to a_t)         → ẑ_{{t+1}}
Decoder head   (ẑ_{{t+1}} → logits_256)                                      → bpb eval
```

**Anti-collapse**: SIGReg (KL penalty toward N(0,I) on encoder outputs).
No EMA, no stop-gradient, no pretrained encoder.

## Why latent actions?

Text has latent "actions" — discourse moves (assertion, elaboration, pivot),
paragraph-level intent, structural patterns (code blocks, citations).
The action encoder captures this slower timescale, giving the predictor
discourse conditioning beyond just local byte history.

## Hyperparameters

- `d_model`: {CFG.d_model}
- `d_action`: {CFG.d_action}
- `n_enc_layers`: {CFG.n_enc_layers}  
- `n_act_layers`: {CFG.n_act_layers}
- `n_pred_layers`: {CFG.n_pred_layers}
- `lambda_sigreg`: {CFG.lambda_sigreg}
- `local_window`: {CFG.local_window}
- `context_window`: {CFG.context_window}
- optimizer: Muon (weight matrices) + AdamW (embeddings/norms)

## Reference

Maes, Lucas et al. "LeWorldModel: Stable End-to-End Joint-Embedding Predictive
Architecture from Pixels." arXiv:2603.19312 (2026).
"""

with open(SUBMISSION_DIR / "README.md", "w") as f:
    f.write(readme.strip())

print("Submission package ready:")
for p in sorted(SUBMISSION_DIR.glob("*")):
    print(f"  {p.name}  ({p.stat().st_size/1e3:.1f} KB)")
print()
print("Next step: git add records/track_non_record_16mb/bytejepa_lewm/")
print("           git commit -m 'ByteJEPA: JEPA-for-text, byte-level, SIGReg'")
print("           gh pr create --title 'ByteJEPA: JEPA-for-text, no tokenizer'")

## Next steps & known limitations

### What to try first if bpb is above baseline
1. **Increase `max_steps`** to 20k+ and use the non-record track (no 10-min limit).
2. **Tune `lambda_sigreg`**: too high → representations collapse to N(0,I); too low → collapse. Grid: `[0.01, 0.05, 0.1, 0.5]`.
3. **Scale `d_model`**: 384 → 512. You have budget headroom from the INT8 compression.
4. **QAT (Quantization-Aware Training)**: add straight-through estimator rounding to the forward pass in the last 500 steps. This is the biggest single win in the leaderboard (`Int6 QAT` entries).
5. **Sliding window eval**: evaluate with stride < seq_len to use full context.

### Theoretical questions this experiments answers
- Does the action encoder (`a_t`) help? → Run ablation in Cell 12.
- Does SIGReg provide stable training without EMA? → Watch `loss_sigreg` curve.
- Does latent-space prediction beat raw byte prediction at matched parameter count?

### Connection to your OMP work
The state encoder is doing **sparse pursuit in latent space**: selecting a low-dimensional support that represents the current byte context. The action encoder selects the **dictionary** (discourse mode). The predictor does **forward pursuit** in that dictionary. If this framing holds, the optimal `d_model` should scale as `O(k log n)` where `k` is the sparsity level of text and `n` is the vocab size — a testable prediction from your SPriFed-OMP theory.